In [1]:
import pandas as pd

In [3]:
from google.colab import files

In [4]:
uploaded=files.upload()

Saving sales_raw_500.xlsx - Sheet1.csv to sales_raw_500.xlsx - Sheet1.csv


In [7]:
df=pd.read_csv('sales_raw_500.xlsx - Sheet1.csv')
df

,Order_ID,Order_Date,Customer,Region,Product,Sales,Cost
0,1001,1/13/2023,Rohit,South,Mobile,23289,19118
1,1002,10/7/2023,Aman,West,Laptop,8905,5593
2,1003,11/5/2023,Rahul,South,Monitor,59987,39959
3,1004,2/19/2024,Rahul,South,Printer,49597,33892
4,1005,1/26/2024,Anjali,North,Laptop,54797,34468
...,...,...,...,...,...,...,...
515,1389,2/12/2024,Vikas,North,Laptop,17924,12145
516,1496,4/24/2023,Kiran,North,Printer,23434,16388
517,1031,2/19/2023,Vikas,East,Printer,58883,43579
518,1317,7/5/2023,Aman,North,Laptop,76781,66865


In [8]:
df.head(3)

,Order_ID,Order_Date,Customer,Region,Product,Sales,Cost
0,1001,1/13/2023,Rohit,South,Mobile,23289,19118
1,1002,10/7/2023,Aman,West,Laptop,8905,5593
2,1003,11/5/2023,Rahul,South,Monitor,59987,39959


In [9]:
df.isnull().sum()

,0
Order_ID,0
Order_Date,18
Customer,0
Region,0
Product,0
Sales,0
Cost,0


In [15]:
df.dropna(subset=["Order_Date"], inplace=True)

In [16]:
df.isnull().sum()

,0
Order_ID,0
Order_Date,0
Customer,0
Region,0
Product,0
Sales,0
Cost,0


In [10]:
df.shape

(520, 7)

In [11]:
df.columns

Index(['Order_ID', 'Order_Date', 'Customer', 'Region', 'Product', 'Sales',
       'Cost'],
      dtype='object')

In [13]:
df.describe()

,Order_ID,Sales,Cost
count,520.000000,520.000000,520.000000
mean,1250.867308,41129.607692,30675.001923
std,145.030140,21605.117122,16598.086873
min,1001.000000,5205.000000,3272.000000
25%,1124.750000,22015.750000,16388.000000
50%,1251.500000,41044.000000,30349.000000
75%,1377.250000,59245.000000,43363.500000
max,1500.000000,79928.000000,71417.000000


#connect from sql

In [19]:
import sqlite3
conn=sqlite3.connect("sales.db")
print("SQLite connected successfully!")


SQLite connected successfully!


#import csv file

In [37]:
df = pd.read_csv("sales_raw_500.xlsx - Sheet1.csv")
df.dropna(subset=["Order_Date"], inplace=True)
df['Order_Date'] = pd.to_datetime(df['Order_Date']).dt.strftime('%Y-%m-%d')
print(df.head())

   Order_ID  Order_Date Customer Region  Product  Sales   Cost
0      1001  2023-01-13    Rohit  South   Mobile  23289  19118
1      1002  2023-10-07     Aman   West   Laptop   8905   5593
2      1003  2023-11-05    Rahul  South  Monitor  59987  39959
3      1004  2024-02-19    Rahul  South  Printer  49597  33892
4      1005  2024-01-26   Anjali  North   Laptop  54797  34468


#connect to sqlite

In [24]:
conn = sqlite3.connect("sales.db")

#import the data into sqlite

In [39]:
df.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

print("Data successfully imported into SQLite!")

Data successfully imported into SQLite!


#check the records

In [26]:
result = pd.read_sql("SELECT COUNT(*) AS total_records FROM sales", conn)
print(result)

   total_records
0            520


#total sales

In [29]:
df = pd.read_sql('SELECT SUM(Sales) AS total_sales FROM sales', conn)
print(result)

   total_sales
0     21387396


#total profit

In [31]:
df = pd.read_sql('SELECT SUM(Sales - Cost) AS total_profit FROM sales', conn)
print(df)

   total_profit
0       5436395


#category wise sales

In [33]:
df_category_sales = pd.read_sql("""
SELECT Product,
       SUM(Sales) AS total_sales
FROM sales
GROUP BY Product
ORDER BY total_sales DESC;
""", conn)
print(df_category_sales)

   Product  total_sales
0  Printer      4819084
1   Tablet      4345390
2   Laptop      4226723
3   Mobile      4033522
4  Monitor      3962677


#monthly sales

In [40]:
df_monthly_sales = pd.read_sql("""
SELECT
    strftime('%Y', Order_Date) AS year,
    strftime('%m', Order_Date) AS month,
    SUM(Sales) AS total_sales
FROM sales
GROUP BY year, month
ORDER BY year, month;
""", conn)
print(df_monthly_sales)

    year month  total_sales
0   None  None       780117
1   2023    01      1045359
2   2023    02      1655381
3   2023    03      1620152
4   2023    04      1021131
5   2023    05      1589894
6   2023    06      1310781
7   2023    07      1729710
8   2023    08      1851789
9   2023    09      1160899
10  2023    10      1391962
11  2023    11      1109145
12  2023    12      1406282
13  2024    01      1383237
14  2024    02      1098031
15  2024    03      1233526


#top ten sales

In [38]:
df_top_products = pd.read_sql("""
SELECT Product,
       SUM(Sales) AS total_sales
FROM sales
GROUP BY Product
ORDER BY total_sales DESC
LIMIT 10;
""", conn)
print(df_top_products)

   Product  total_sales
0  Printer      4819084
1   Tablet      4345390
2   Laptop      4226723
3   Mobile      4033522
4  Monitor      3962677
